In [20]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load matches
matches = pd.read_csv('clean_matches.csv')
matches.columns = matches.columns.str.lower().str.strip()
matches['matchid'] = matches['matchid'].astype(str).str.strip()
matches['date'] = pd.to_datetime(matches['date'], errors='coerce')

print("Matches shape:", matches.shape)
print("Matches columns:", matches.columns.tolist())

# Load deliveries
deliveries = pd.read_csv('clean_deliveries.csv')
deliveries.columns = deliveries.columns.str.lower().str.strip()
deliveries['matchid'] = deliveries['matchid'].astype(str).str.strip()

print("\nDeliveries shape:", deliveries.shape)
print("Deliveries columns:", deliveries.columns.tolist())

# Merge venue, season, date, teams from matches
match_cols = [c for c in ['matchid', 'venue', 'season', 'date', 'team1', 'team2', 'winner'] if c in matches.columns]
deliveries = deliveries.merge(matches[match_cols], on='matchid', how='left')

# Fill missing columns safely
for col, default in [
    ('venue', 'Unknown'),
    ('season', 'Unknown'),
    ('team1', 'Unknown'),
    ('team2', 'Unknown'),
    ('winner', 'No Result')
]:
    if col in deliveries.columns:
        deliveries[col] = deliveries[col].fillna(default)
    else:
        deliveries[col] = default

if 'date' in deliveries.columns:
    deliveries['date'] = pd.to_datetime(deliveries['date'], errors='coerce')
else:
    deliveries['date'] = pd.NaT

print("\nAfter merge - key columns:", [c for c in ['venue','season','date','team1','team2'] if c in deliveries.columns])

Matches shape: (1169, 20)
Matches columns: ['season', 'venue', 'event', 'winner_runs', 'umpire2', 'toss_winner', 'date', 'umpire1', 'city', 'reserve_umpire', 'winner', 'team1', 'toss_decision', 'team2', 'winner_wickets', 'tv_umpire', 'player_of_match', 'match_referee', 'match_number', 'matchid']

Deliveries shape: (278352, 37)
Deliveries columns: ['matchid', 'inning', 'over_ball', 'over', 'ball', 'batting_team', 'bowling_team', 'batsman', 'non_striker', 'bowler', 'batsman_runs', 'extras', 'iswide', 'isnoball', 'byes', 'legbyes', 'penalty', 'dismissal_kind', 'player_dismissed', 'date', 'match_id', 'season', 'start_date', 'venue', 'innings', 'striker', 'runs_off_bat', 'wides', 'noballs', 'byes.1', 'legbyes.1', 'penalty.1', 'wicket_type', 'other_wicket_type', 'other_player_dismissed', 'total_runs', 'is_valid_ball']

After merge - key columns: ['venue', 'season', 'date', 'team1', 'team2']


In [21]:
# Aggregate valid balls only
valid_balls = deliveries[deliveries.get('is_valid_ball', 0) == 1]

batsman_innings = valid_balls.groupby(['matchid', 'inning', 'batsman']).agg(
    runs_scored=('batsman_runs', 'sum'),
    balls_faced=('ball', 'count')
).reset_index()

# Merge back match-level info (batting_team, bowling_team, venue, etc.)
info_cols = ['matchid', 'inning', 'batsman']
for c in ['batting_team', 'bowling_team', 'venue', 'season', 'date', 'team1', 'team2']:
    if c in deliveries.columns:
        info_cols.append(c)

match_info = deliveries[info_cols].drop_duplicates(subset=['matchid', 'inning', 'batsman'])

batsman_innings = batsman_innings.merge(match_info, on=['matchid', 'inning', 'batsman'], how='left')

# Current innings strike rate
batsman_innings['strike_rate'] = np.where(
    batsman_innings['balls_faced'] > 0,
    (batsman_innings['runs_scored'] / batsman_innings['balls_faced']) * 100,
    0
)

# Sort for historical calculation
if 'date' in batsman_innings.columns:
    batsman_innings['date'] = pd.to_datetime(batsman_innings['date'], errors='coerce')
    batsman_innings = batsman_innings.sort_values(['batsman', 'date']).reset_index(drop=True)
else:
    batsman_innings = batsman_innings.sort_values(['batsman', 'matchid']).reset_index(drop=True)

print("Batsman-innings shape:", batsman_innings.shape)
print("Columns:", batsman_innings.columns.tolist())

Batsman-innings shape: (17700, 13)
Columns: ['matchid', 'inning', 'batsman', 'runs_scored', 'balls_faced', 'batting_team', 'bowling_team', 'venue', 'season', 'date', 'team1', 'team2', 'strike_rate']


In [22]:
# Historical features
batsman_innings['innings_played'] = batsman_innings.groupby('batsman').cumcount() + 1

batsman_innings['batsman_avg'] = batsman_innings.groupby('batsman')['runs_scored'].transform(
    lambda x: x.shift(1).expanding().mean().fillna(0)
)

batsman_innings['balls_faced_avg'] = batsman_innings.groupby('batsman')['balls_faced'].transform(
    lambda x: x.shift(1).expanding().mean().fillna(0)
)

cum_runs = batsman_innings.groupby('batsman')['runs_scored'].cumsum().shift(1).fillna(0)
cum_balls = batsman_innings.groupby('batsman')['balls_faced'].cumsum().shift(1).fillna(0)
batsman_innings['historical_sr'] = np.where(cum_balls > 0, (cum_runs / cum_balls) * 100, 0)

# Avg vs opponent
temp = batsman_innings.sort_values(['batsman', 'bowling_team', 'matchid'])
temp['avg_vs_opp'] = temp.groupby(['batsman', 'bowling_team'])['runs_scored'].transform(
    lambda x: x.shift(1).expanding().mean().fillna(0)
)
batsman_innings = temp.sort_values(['batsman', 'matchid']).reset_index(drop=True)

# Opposition strength
innings_totals = deliveries.groupby(['matchid', 'inning'])['total_runs'].sum().reset_index(name='innings_total')
innings_totals = innings_totals.merge(
    deliveries[['matchid', 'inning', 'bowling_team']].drop_duplicates(),
    on=['matchid', 'inning']
)
innings_totals['opp_strength'] = innings_totals.groupby('bowling_team')['innings_total'].transform(
    lambda x: x.shift(1).expanding().mean().fillna(innings_totals['innings_total'].mean())
)

batsman_innings = batsman_innings.merge(
    innings_totals[['matchid', 'inning', 'opp_strength']],
    on=['matchid', 'inning'],
    how='left'
)
batsman_innings['opp_strength'] = batsman_innings['opp_strength'].fillna(batsman_innings['opp_strength'].mean())

# Home advantage
if 'team1' in batsman_innings.columns:
    batsman_innings['home_advantage'] = (batsman_innings['batting_team'] == batsman_innings['team1']).astype(int)
else:
    batsman_innings['home_advantage'] = 0

# Season year
if 'season' in batsman_innings.columns:
    batsman_innings['season_year'] = batsman_innings['season'].astype(str).str.extract(r'(\d{4})').astype(float)
    batsman_innings['season_year'] = batsman_innings['season_year'].fillna(batsman_innings['date'].dt.year if 'date' in batsman_innings.columns else 2020)
else:
    batsman_innings['season_year'] = 2020

# Venue encoding
le = LabelEncoder()
batsman_innings['venue_encoded'] = le.fit_transform(batsman_innings['venue'])

# Filter & save
processed = batsman_innings[batsman_innings['innings_played'] > 5].copy()

processed.to_csv('processed_player_data.csv', index=False)
import joblib
joblib.dump(le, 'venue_encoder.pkl')

print("\nSaved processed_player_data.csv and venue_encoder.pkl")
print("Shape:", processed.shape)
print("Columns:", processed.columns.tolist())
print("Sample features + target:\n", processed[['batsman','runs_scored','batsman_avg','historical_sr','avg_vs_opp','venue_encoded','opp_strength','home_advantage','season_year']].head(3))


Saved processed_player_data.csv and venue_encoder.pkl
Shape: (14894, 22)
Columns: ['matchid', 'inning', 'batsman', 'runs_scored', 'balls_faced', 'batting_team', 'bowling_team', 'venue', 'season', 'date', 'team1', 'team2', 'strike_rate', 'innings_played', 'batsman_avg', 'balls_faced_avg', 'historical_sr', 'avg_vs_opp', 'opp_strength', 'home_advantage', 'season_year', 'venue_encoded']
Sample features + target:
           batsman  runs_scored  batsman_avg  historical_sr  avg_vs_opp  \
5  A Ashish Reddy          7.0          7.0     120.689655         0.0   
6  A Ashish Reddy         14.0          7.0     127.272727         4.0   
7  A Ashish Reddy         16.0          8.0     124.444444         0.0   

   venue_encoded  opp_strength  home_advantage  season_year  
5              0    144.600000               0          NaN  
6              0    158.822222               0          NaN  
7              0    157.257143               0          NaN  
